# 🔴 KaizenStat — Advanced Demo (30 min)

**Level:** Advanced | **Time:** ~30 minutes | **Dataset:** Titanic + engineered features

This notebook covers **advanced KaizenStat techniques**:

| Topic | Method | What it unlocks |
|-------|--------|-----------------|
| Feature engineering | Manual + doctor pipeline | Better model inputs |
| Hyperparameter tuning | `train(tune=True)` | +5–15% score improvement |
| Custom models | `add_model()` | Plug in any sklearn-compatible model |
| Custom checks | `add_check()` | Domain-specific validation rules |
| Feature impact | `feature_impact()` | Counterfactual importance |
| Dataset difficulty | `dataset_difficulty()` | Understand ceiling score |
| Code generation | `codegen()` | Export a standalone, production-ready script |
| Drift detection | `detect_drift()` | Monitor train vs test distribution |
| Auto-improve loop | `auto_improve(tune=True)` | Automated fix + retrain cycle |

---
> **Prerequisites:** You've completed the Intermediate demo
>
> **Goal:** Get the highest possible Titanic accuracy using every KaizenStat tool available

In [ ]:
!pip install kaizenstat -q
print("✅ KaizenStat installed")

## Setup — Load + Feature Engineering

We'll add domain-knowledge features before running the pipeline.
KaizenStat works best when you bring your domain knowledge to the data prep.

In [ ]:
import pandas as pd
import numpy as np
from kaizenstat import DataDoctor

# Load raw Titanic data
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df_raw = pd.read_csv(url)

print(f"Raw data: {df_raw.shape}")

# --- Feature Engineering ---
df = df_raw.copy()

# 1. Title extraction from Name
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.')
title_map = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mlle': 'Miss', 'Countess': 'Rare', 'Ms': 'Miss', 'Lady': 'Rare',
    'Jonkheer': 'Rare', 'Don': 'Rare', 'Dona': 'Rare', 'Mme': 'Mrs',
    'Capt': 'Rare', 'Sir': 'Rare'
}
df['Title'] = df['Title'].map(title_map).fillna('Rare')

# 2. Family size
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# 3. Deck from Cabin
df['Deck'] = df['Cabin'].str[0].fillna('Unknown')

# 4. Age band (fill missing first)
df['Age'] = df['Age'].fillna(df['Age'].median())
df['AgeBand'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100],
                        labels=['Child', 'Teen', 'Adult', 'MidAge', 'Senior'])

# 5. Fare per person
df['FarePerPerson'] = df['Fare'] / df['FamilySize'].replace(0, 1)

# Drop original columns we've extracted from
df = df.drop(columns=['Name', 'Ticket', 'Cabin', 'PassengerId'])

print(f"Engineered data: {df.shape}")
print(f"New columns: Title, FamilySize, IsAlone, Deck, AgeBand, FarePerPerson")
df.head()

## 1. Dataset Difficulty

Before tuning, understand the ceiling. `dataset_difficulty()` estimates how inherently hard this prediction task is.
- 0.0 = trivially easy (any model gets 99%+)
- 0.5 = moderate challenge
- 1.0 = near-impossible (random noise)

In [ ]:
doctor = DataDoctor()
doctor.fit(df, target='Survived')

difficulty = doctor.dataset_difficulty()
print(f"Dataset difficulty: {difficulty:.3f}")
if difficulty > 0.5:
    print("→ Hard dataset. Focus on feature engineering, not model tuning.")
elif difficulty > 0.3:
    print("→ Moderate difficulty. Tuning will help.")
else:
    print("→ Easy dataset. Any model should work well.")

## 2. Health + Custom Validation

In [ ]:
health = doctor.health()
print(f"Health Score: {health.score} / 100")

In [ ]:
# Add a domain-specific validation check
def check_survival_rate_by_sex(df, target):
    """Domain check: sex should be a strong predictor on Titanic."""
    if 'Sex' not in df.columns:
        return ["Missing 'Sex' column — known strong predictor for Titanic"]
    male_survival = df[df['Sex'] == 'male'][target].mean()
    female_survival = df[df['Sex'] == 'female'][target].mean()
    if abs(female_survival - male_survival) < 0.3:
        return [f"Unexpected: sex gap is only {abs(female_survival - male_survival):.2f} — check encoding"]
    return []  # No issues found

def check_class_survival_pattern(df, target):
    """Domain check: higher Pclass (lower number) should have higher survival."""
    if 'Pclass' not in df.columns:
        return ["Missing 'Pclass' column"]
    survival_by_class = df.groupby('Pclass')[target].mean()
    if survival_by_class[1] < survival_by_class[3]:
        return ["Unexpected: 1st class has lower survival than 3rd — data issue?"]
    return []

doctor.add_check(check_survival_rate_by_sex, name="sex_survival_check")
doctor.add_check(check_class_survival_pattern, name="class_survival_check")

validation = doctor.validate()

## 3. Drift Detection

Check if training and test distributions are similar using KS test.
This simulates what you'd do in production monitoring.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Survived'])
y = df['Survived']
X_train, X_test, _, _ = train_test_split(X, y, test_size=0.2, random_state=42)

# Select only numeric columns for drift detection
num_cols = X_train.select_dtypes(include='number').columns.tolist()
drift = doctor.detect_drift(X_train[num_cols], X_test[num_cols])

print("Drift detection (KS test p-values for drifting features):")
if drift:
    for col, pval in sorted(drift.items(), key=lambda x: x[1]):
        print(f"  {col:20s}  p={pval:.4f}  ⚠️ DRIFT" if pval < 0.05 else f"  {col:20s}  p={pval:.4f}")
else:
    print("  ✅ No significant drift detected")

## 4. Fix Data

In [ ]:
fixed_df = doctor.fix(safe=True)
print(f"Fixed: {fixed_df.isnull().sum().sum()} missing values remaining")

## 5. Custom Model — Add Your Own

`add_model()` lets you inject any sklearn-compatible estimator into the benchmark.
Your model competes against KaizenStat's built-in candidates.

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier

# Add custom models to the benchmark
doctor.add_model("SVM_RBF", SVC(kernel='rbf', probability=True, C=1.0, random_state=42))
doctor.add_model("ExtraTrees", ExtraTreesClassifier(n_estimators=100, random_state=42))

print("Custom models registered. They will compete in the next train() call.")

## 6. Train with Tuning

`tune=True` runs `RandomizedSearchCV` on the best model found during benchmarking.
- `n_iter=30` — number of random configurations to try
- Typical gain: +3–10% over untuned best model

**Tip:** Start with `n_iter=20` for speed. Use `n_iter=50` for maximum accuracy.

In [ ]:
print("Training with hyperparameter tuning (this takes ~2–3 min)...")
train_result = doctor.train(cv=5, tune=True, n_iter=30)

print(f"\n{'='*50}")
print("TUNED TRAINING RESULTS")
print(f"{'='*50}")
print(f"Best model:   {train_result.model_name}")
print(f"Test score:   {train_result.test_score:.4f}")
print(f"Train score:  {train_result.train_score:.4f}")
if hasattr(train_result, 'best_params') and train_result.best_params:
    print(f"Best params:  {train_result.best_params}")

## 7. Compare: Baseline vs Tuned vs Engineered

Now let's compare raw data vs feature-engineered data side by side.

In [ ]:
# Baseline (raw data, no tuning)
df_raw_clean = pd.read_csv(url)
d_base = DataDoctor()
d_base.fit(df_raw_clean, target='Survived')
d_base.fix(safe=True)
base = d_base.train(cv=5, tune=False)

# Our engineered + tuned version
tuned_score = train_result.test_score

print(f"\n{'='*50}")
print("COMPARISON")
print(f"{'='*50}")
print(f"Baseline (raw, no tuning):  {base.test_score:.4f}")
print(f"Engineered + tuned:         {tuned_score:.4f}")
delta = tuned_score - base.test_score
print(f"Improvement:                {delta:+.4f} ({delta*100:.1f}%)")

## 8. Deep Debug — Feature Impact + Failure Slices

In [ ]:
debug_result = doctor.debug_model()

print(f"\nTest score:  {debug_result.test_score:.4f}")
print(f"Gap:         {debug_result.gap:.4f}")

In [ ]:
print("\nCounterfactual Feature Impact (score drop when each feature is removed):")
print("Higher = more important to the model\n")

impact = doctor.feature_impact(top_n=15)
sorted_impact = sorted(impact.items(), key=lambda x: -x[1])

for feat, drop in sorted_impact:
    bar = '█' * max(1, int(drop * 200))
    print(f"  {feat:25s}  {drop:.4f}  {bar}")

# Feature engineering insight
print("\n--- Insight ---")
top_feat = sorted_impact[0][0] if sorted_impact else "unknown"
print(f"Most important feature: {top_feat}")
print("If 'Sex' or 'Title' is top, domain knowledge helped the model.")
print("If 'FamilySize' or 'FarePerPerson' is top, our feature engineering added value.")

## 9. Improvement Suggestions

In [ ]:
improvement_report = doctor.improve()

print("\n--- Prioritised action list ---")
actions = doctor.recommend_actions()
for i, action in enumerate(actions, 1):
    print(f"  {i}. {action}")

## 10. Trust Score — Production Readiness

In [ ]:
trust = doctor.trust_score()

print(f"\n{'='*50}")
print("PRODUCTION READINESS ASSESSMENT")
print(f"{'='*50}")
print(f"Trust Score: {trust.score:.0f} / 100")

thresholds = [(90, "✅ Excellent — safe to deploy to production"),
              (75, "✅ Good — production ready with monitoring"),
              (60, "🟡 Fair — needs improvement before production"),
              (0,  "🔴 Poor — not production ready")]

for threshold, message in thresholds:
    if trust.score >= threshold:
        print(f"Verdict: {message}")
        break

confidence = doctor.pipeline_confidence()
print(f"\nPipeline Confidence: {confidence} / 100")

## 11. Code Generation — Production-Ready Script

`codegen()` exports a **standalone Python script** that reproduces your entire pipeline.
- **No KaizenStat required at runtime** — only pure scikit-learn
- Ready to drop into a Flask/FastAPI service
- Includes all preprocessing steps

In [ ]:
script_path = doctor.codegen(output_path="titanic_pipeline.py")
print(f"✅ Standalone pipeline script: {script_path}")

# Preview the generated code
with open(script_path, 'r') as f:
    code = f.read()

print("\n--- Generated script preview (first 40 lines) ---")
lines = code.split('\n')
for i, line in enumerate(lines[:40], 1):
    print(f"{i:3d}  {line}")
if len(lines) > 40:
    print(f"... ({len(lines) - 40} more lines)")

## 12. Export Model + Report

In [ ]:
# Export trained model
model_path = doctor.export_model(path="titanic_advanced_model.joblib")
print(f"✅ Model exported: {model_path}")

# Generate full HTML report
report_path = doctor.report(output_path="advanced_titanic_report.html")
print(f"✅ Report exported: {report_path}")

# Display report
from IPython.display import IFrame, display
display(IFrame(src='advanced_titanic_report.html', width='100%', height='700px'))

## 13. Production Inference — Loading Your Exported Model

Simulate what happens in production when you load the model and predict on new data.

In [ ]:
import joblib

# Load the exported model
pipeline = joblib.load(model_path)
print(f"Model loaded: {type(pipeline).__name__}")

# Create sample passengers (new data never seen by the model)
sample_passengers = pd.DataFrame([
    # First class woman — likely survived
    {'Pclass': 1, 'Sex': 'female', 'Age': 35.0, 'SibSp': 1, 'Parch': 0,
     'Fare': 80.0, 'Embarked': 'S', 'Title': 'Mrs',
     'FamilySize': 2, 'IsAlone': 0, 'Deck': 'C',
     'AgeBand': 'Adult', 'FarePerPerson': 40.0},
    # Third class man — likely did not survive
    {'Pclass': 3, 'Sex': 'male', 'Age': 22.0, 'SibSp': 0, 'Parch': 0,
     'Fare': 7.5, 'Embarked': 'S', 'Title': 'Mr',
     'FamilySize': 1, 'IsAlone': 1, 'Deck': 'Unknown',
     'AgeBand': 'Adult', 'FarePerPerson': 7.5},
])

# Predict
predictions = pipeline.predict(sample_passengers)
print(f"\nPredictions:")
for i, (pred, row) in enumerate(zip(predictions, sample_passengers.itertuples())):
    label = 'SURVIVED' if pred == 1 else 'DID NOT SURVIVE'
    sex = sample_passengers.iloc[i]['Sex']
    pclass = sample_passengers.iloc[i]['Pclass']
    print(f"  Passenger {i+1} (Class {pclass}, {sex}): {label}")

## Summary — What We Built

```python
# Everything we did in this notebook
doctor = DataDoctor()
doctor.fit(df, target='Survived')          # fit with feature-engineered data
doctor.dataset_difficulty()                 # understand ceiling score
doctor.health()                             # check data quality
doctor.add_check(custom_fn, name='...')    # domain-specific validation
doctor.validate()                           # leakage + drift + custom checks
doctor.detect_drift(X_train, X_test)       # monitor distribution drift
doctor.fix(safe=True)                       # auto-heal data
doctor.add_model('SVM', SVC(...))          # plug in custom model
doctor.train(tune=True, n_iter=30)         # benchmark + tune best model
doctor.debug_model()                        # root-cause failure analysis
doctor.feature_impact(top_n=15)            # counterfactual feature importance
doctor.improve()                            # ranked improvement suggestions
doctor.recommend_actions()                  # structured next-steps
doctor.trust_score()                        # production readiness (0–100)
doctor.pipeline_confidence()               # holistic pipeline score
doctor.codegen(output_path='pipeline.py') # standalone production script
doctor.export_model(path='model.joblib')  # export trained model
doctor.report()                             # full HTML report
```

### Key Insights from This Notebook

1. **Feature engineering mattered more than tuning** — `Title`, `FamilySize`, `Deck` improved the model more than hyperparameter search
2. **Custom checks caught domain issues** — sex and class survival patterns validated our data integrity
3. **Codegen exports pure sklearn** — no KaizenStat dependency in production
4. **Trust Score > 75 = production ready** — accuracy alone is not enough

## 🎯 Further Challenges

**Challenge 1:** Can you break 85% accuracy?
```python
# Try adding interaction features:
df['Sex_Pclass'] = df['Sex'] + '_' + df['Pclass'].astype(str)
df['Age_Pclass'] = df['Age'] * df['Pclass']
```

**Challenge 2:** Try the auto_improve loop
```python
doctor_auto = DataDoctor()
doctor_auto.fit(df, target='Survived')
comparison = doctor_auto.auto_improve(tune=True)
print(f"Auto-improve delta: {comparison.score_delta:+.4f}")
```

**Challenge 3:** Use KaizenStat on a different Kaggle dataset
- [House Prices](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques) — regression
- [Credit Card Fraud](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) — imbalanced classification
- [Amazon Reviews](https://www.kaggle.com/datasets/bittlingmayer/amazonreviews) — text/NLP mode

**Challenge 4:** Monitor production drift
```python
# Simulate production: score each week's data against training distribution
# If drift detected → retrigger doctor.fit() → doctor.train()
```

---
## Notebook Series

| Notebook | Level | Time |
|----------|-------|------|
| [Basic](demo_basic.ipynb) | 🟢 Beginner | 5 min |
| [Intermediate](demo_intermediate.ipynb) | 🟡 Intermediate | 15 min |
| **You are here** | 🔴 Advanced | 30 min |
| [Quick Start](quickstart_tabular.ipynb) | ⚡ Reference | 2 min |

---
*KaizenStat v0.5.1 · [GitHub](https://github.com/masuddarrahaman/KaizenStat-Library) · MIT License*

*Built by [Masuddar Rahaman](https://github.com/masuddarrahaman)*